In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import os
import seaborn as sns

In [ ]:
# Task 1: Write your code here:
dataset = pd.read_csv(os.path.join(path, "Q1_data.csv"))

In [ ]:
# Task 2: Write your code here:
dataset.head(10)

In [ ]:
# Task 3: Write your code here:
dataset.info()

In [ ]:
# Task 4: Write your code here:
dataset.describe()

In [ ]:
# Task 5: Write your code here:
sns.histplot(dataset.Delivery_Time)
# data seems to have a small right skewe

In [ ]:
# Task 1: Write your code here:
clean_data = dataset.copy()

clean_data = clean_data.drop(columns='Order_ID')
list(clean_data.columns)

In [ ]:
# Task 2: Write your code here:
clean_data.isna().sum()

In [ ]:
clean_data.Weather.describe()

In [ ]:
print(clean_data.Weather.value_counts(normalize=True, dropna=False))
# clean_data = clean_data.dropna(subset='Weather')
# less than 5%, we can drop it
# or fill with mode
clean_data['Weather']= clean_data['Weather'].fillna(clean_data.Weather.mode()[0])
clean_data.Weather.value_counts(normalize=True, dropna=False)


In [ ]:
clean_data.Traffic_Level.describe()

In [ ]:
clean_data.Traffic_Level.value_counts(normalize=True, dropna=False)
clean_data = clean_data.dropna(subset='Traffic_Level')
# less than 5%, we can drop it
clean_data.Traffic_Level.value_counts(normalize=True, dropna=False)


In [ ]:
# Task 3: Write your code here:
print(clean_data.Time_of_Day.value_counts(normalize=True, dropna=False))
clean_data = clean_data.dropna(subset='Time_of_Day')
# less than 5%, we can drop it
clean_data.Traffic_Level.value_counts(normalize=True, dropna=False)


In [ ]:
clean_data.Courier_Experience_yrs.describe()
# Low std, but relativily good

In [ ]:
# Task 4: Write your code here:
print(clean_data.Courier_Experience_yrs.value_counts(normalize=True, dropna=False))
# less than 5%, we can drop it
# or imput mean
clean_data['Courier_Experience_yrs']= clean_data['Courier_Experience_yrs'].fillna(clean_data.Courier_Experience_yrs.mean())
clean_data.Courier_Experience_yrs.value_counts(normalize=True, dropna=False)


In [ ]:
# NAN Target features are usless
clean_data.dropna(subset='Delivery_Time', inplace=True)



In [ ]:
print(clean_data.duplicated().sum()) # A LOT !
clean_data.drop_duplicates(inplace=True)
print(clean_data.duplicated().sum())


In [ ]:

clean_data = pd.get_dummies(clean_data, drop_first=True) # OHE but with pandas :)
clean_data.info()


In [ ]:
from sklearn.model_selection import train_test_split
# # split befor scaling, to avoid info leake (STD, Mean)

# Xtrain, Xtest, ytrain, ytest = train_test_split(clean_data.drop(columns='Delivery_Time'),clean_data['Delivery_Time'] , test_size=.2, shuffle= True, random_state=11)
# print(Xtrain.shape)
# print(ytrain.shape)
# print(Xtest.shape)
# print(ytest.shape)

# scale target?
# we can use inverse transform for the target. scale_pred -> scaler.inverse() -> real value
# but I will skip this for time



In [ ]:
from sklearn.preprocessing import StandardScaler

s = StandardScaler()
num_cols = clean_data.select_dtypes(include= 'number').drop(columns='Delivery_Time').columns

clean_data[num_cols] = s.fit_transform(clean_data[num_cols])
# clean_data[num_cols] = s.transform(clean_data[num_cols])



In [ ]:
# Task 6: Write your code here:
# target is numerical


In [ ]:
# Task 1: Write your code here:
# already did it

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Storage for linear regression results for each fold
maes = []

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

X = clean_data.drop(columns='Delivery_Time')
y = clean_data['Delivery_Time']
model_name='RF'
model = RandomForestRegressor(max_depth=15, n_estimators=200)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  print(f"Training {model_name}...")

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test,y_pred)
  maes.append(mae)
  print(f'MAE = {mae:.3f}')
    # Store results
print(f'Avg MAE = {np.mean(mae):.3f}')

In [ ]:
# Task 1: Write your code here:

fi = model.feature_importances_
# fi_df = pd.DataFrame(columns = clean_data.drop(columns='Delivery_Time').columns,data= fi)
sns.barplot(x=clean_data.drop(columns='Delivery_Time').columns, y=fi)
#  need some edits to be more readble

In [ ]:
# Task 2: Write your code here:
preds = model.predict(clean_data.drop(columns='Delivery_Time'))
sns.histplot(preds)

In [ ]:
# compare with actual
sns.histplot(clean_data.Delivery_Time)

In [ ]:
# Task Bonus: Write your code here: